# StyleGAN2-ADA per Facial Expression Recognition (FER)

Questo notebook mostra **il workflow corretto** per usare **StyleGAN2-ADA pre-addestrata**
al fine di generare immagini sintetiche di espressioni facciali e migliorare l'accuracy FER.

**Nota**: l'addestramento reale viene fatto tramite l'implementazione ufficiale NVIDIA.

## 1. Setup ambiente (Colab)

In [ ]:
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch
!pip install -r requirements.txt

## 2. Download modello pre-addestrato (FFHQ)

In [ ]:
!wget https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan2-ada-pytorch/versions/1/files/ffhq.pkl

In [ ]:
# Installazione dipendenze per classificazione età
!pip install deepface opencv-python-headless h5py pillow

In [ ]:
import h5py
import numpy as np
import os

# Path al dataset h5 (modifica se necessario)
BASE_DIR = os.path.expanduser('~')
DATA_DIR = os.path.join(BASE_DIR, 'data')
DATASET_PATH = os.path.join(DATA_DIR, 'dataset.h5')

# Ispeziona struttura del file h5
print('=== Struttura dataset h5 ===')
with h5py.File(DATASET_PATH, 'r') as f:
    for key in f.keys():
        item = f[key]
        if hasattr(item, 'shape'):
            print(f'{key}: shape={item.shape}, dtype={item.dtype}')
        else:
            print(f'{key}: {type(item)}')
    
    # Verifica metadati età
    has_age_metadata = False
    if 'age' in f.keys():
        print('\n✓ Trovato campo "age"')
        has_age_metadata = True
    elif 'is_adult' in f.keys():
        print('\n✓ Trovato campo "is_adult"')
        has_age_metadata = True
    elif 'metadata' in f.keys():
        print('\n✓ Trovato campo "metadata"')
        has_age_metadata = True
    else:
        print('\n⚠️ Nessun metadato età trovato nel h5.')
        print('Useremo DeepFace per classificare età automaticamente.')
    
    # Carica dati base
    X_train = np.array(f['X_train'])
    y_train = np.array(f['y_train'])
    X_val = np.array(f['X_val'])
    y_val = np.array(f['y_val'])
    class_names = [c.decode('utf-8') for c in f['class_names']]

X = np.concatenate([X_train, X_val])
y = np.concatenate([y_train, y_val])

print(f'\nDataset completo: {X.shape}')
print(f'Classi: {class_names}')

In [ ]:
from deepface import DeepFace
from PIL import Image
from tqdm import tqdm
import gc

# Parametri
ADULT_AGE_THRESHOLD = 18  # Consideriamo adulti chi ha >= 18 anni
BATCH_SIZE = 100  # Processa in batch per evitare memory overflow

# Funzione per classificare età usando DeepFace
def is_adult(img_array):
    """
    Classifica un'immagine come adulto/bambino.
    Returns: True se adulto, False altrimenti (o errore)
    """
    try:
        # DeepFace richiede formato (H, W, 3)
        result = DeepFace.analyze(
            img_path=img_array,
            actions=['age'],
            detector_backend='opencv',
            enforce_detection=False  # Non fallisce se non trova viso
        )
        
        if isinstance(result, list):
            result = result[0]
        
        age = result.get('age', 0)
        return age >= ADULT_AGE_THRESHOLD
    except:
        # In caso di errore, scarta l'immagine (conservativo)
        return False

# Filtra dataset: mantieni solo adulti
print('Inizio classificazione età (può richiedere tempo)...')
adult_mask = np.zeros(len(X), dtype=bool)

for i in tqdm(range(len(X)), desc='Classificando età'):
    if is_adult(X[i]):
        adult_mask[i] = True
    
    # Libera memoria periodicamente
    if (i + 1) % BATCH_SIZE == 0:
        gc.collect()

X_adults = X[adult_mask]
y_adults = y[adult_mask]

print(f'\n=== Risultati Filtro ===')
print(f'Immagini originali: {len(X)}')
print(f'Immagini adulti: {len(X_adults)} ({100*len(X_adults)/len(X):.1f}%)')
print(f'Immagini bambini scartate: {len(X) - len(X_adults)} ({100*(1-len(X_adults)/len(X)):.1f}%)')

# Distribuzione per classe dopo il filtro
print('\nDistribuzione per classe (solo adulti):')
for i, name in enumerate(class_names):
    count = np.sum(y_adults == i)
    count_orig = np.sum(y == i)
    print(f'{name:12s}: {count:5d} / {count_orig:5d} originali ({100*count/count_orig if count_orig > 0 else 0:.1f}%)')

In [ ]:
# Salva dataset filtrato (solo adulti) in formato h5
OUTPUT_FILTERED_H5 = os.path.join(DATA_DIR, 'dataset_adults_only.h5')

# Risplittiamo in train/val mantenendo proporzioni
from sklearn.model_selection import train_test_split

X_train_adults, X_val_adults, y_train_adults, y_val_adults = train_test_split(
    X_adults, y_adults, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_adults
)

# Salva nuovo h5 con solo adulti
with h5py.File(OUTPUT_FILTERED_H5, 'w') as f:
    f.create_dataset('X_train', data=X_train_adults, compression='gzip')
    f.create_dataset('y_train', data=y_train_adults, compression='gzip')
    f.create_dataset('X_val', data=X_val_adults, compression='gzip')
    f.create_dataset('y_val', data=y_val_adults, compression='gzip')
    f.create_dataset('class_names', data=[c.encode('utf-8') for c in class_names])

print(f'\n✓ Salvato dataset filtrato (solo adulti): {OUTPUT_FILTERED_H5}')
print(f'Train: {X_train_adults.shape}')
print(f'Val: {X_val_adults.shape}')

In [ ]:
from PIL import Image
import shutil

# Esporta ogni classe in cartelle separate (formato compatibile StyleGAN)
OUTPUT_BASE_DIR = os.path.join(BASE_DIR, 'stylegan_data_adults')
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)

print('\n=== Esportazione classi per StyleGAN ===')

for class_idx, class_name in enumerate(class_names):
    # Filtra immagini della classe corrente
    class_mask = y_adults == class_idx
    class_images = X_adults[class_mask]
    
    if len(class_images) == 0:
        print(f'⚠️ Classe {class_name}: nessuna immagine adulta, skip')
        continue
    
    # Crea cartella per la classe
    class_dir = os.path.join(OUTPUT_BASE_DIR, class_name)
    os.makedirs(class_dir, exist_ok=True)
    
    # Salva immagini come PNG
    for i, img_array in enumerate(class_images):
        img = Image.fromarray(img_array.astype('uint8'))
        img_path = os.path.join(class_dir, f'{i:06d}.png')
        img.save(img_path)
    
    print(f'✓ {class_name:12s}: {len(class_images):5d} immagini → {class_dir}')

print(f'\n✓ Esportazione completata!')
print(f'Cartelle salvate in: {OUTPUT_BASE_DIR}')
print(f'\nProssimi passi:')
print(f'1. Per ogni classe rara, crea lo zip: python dataset_tool.py --source={OUTPUT_BASE_DIR}/DISGUST --dest=DISGUST.zip')
print(f'2. Fine-tune StyleGAN con quella classe')
print(f'3. Genera immagini sintetiche')

### Alternative: Filtro Manuale o Basato su Modello Custom

Se DeepFace è troppo lento o impreciso, puoi:

**Opzione A - Usa modello custom (più veloce):**
```python
# Usa un modello CNN leggero addestrato su classificazione età (es. UTKFace dataset)
# Esempio: https://github.com/yu4u/age-gender-estimation
```

**Opzione B - Se hai metadata nel h5:**
```python
# Se il tuo h5 ha campo 'age' o 'is_adult'
with h5py.File(DATASET_PATH, 'r') as f:
    ages = np.array(f['age'])  # o is_adult
adult_mask = ages >= 18  # filtra direttamente
```

**Opzione C - Filtro manuale basato su dataset source:**
Se conosci il dataset originale (es. RAF-DB, FER2013, AffectNet) e quali split contengono bambini vs adulti, filtra a monte durante la creazione del h5.

## 3. Preparazione dataset FER

Il dataset deve essere organizzato in una cartella contenente **solo immagini di una classe**
(es. DISGUST).

In [ ]:
# Esempio (da adattare al tuo dataset)
# !python dataset_tool.py --source=/content/DISGUST --dest=/content/DISGUST.zip

## 4. Fine-tuning StyleGAN2-ADA (classe rara)

In [ ]:
!python train.py \
  --outdir=training-runs \
  --data=/content/DISGUST.zip \
  --gpus=1 \
  --cfg=auto \
  --resume=ffhq.pkl \
  --aug=ada \
  --kimg=300

## 5. Generazione immagini sintetiche

In [ ]:
!python generate.py \
  --network=training-runs/*/network-snapshot.pkl \
  --seeds=0-999 \
  --outdir=generated

## 6. Uso delle immagini generate

Le immagini generate vanno:
- filtrate con il classificatore FER (confidenza alta)
- unite al dataset originale
- usate per il ri-addestramento finale